In [1]:
print("Hello world")

Hello world


In [2]:
import pandas as pd
import numpy as np
import scipy as sp

In [3]:
from sklearn.model_selection import train_test_split

In [4]:
%pip freeze

asttokens==3.0.1
click==8.3.1
colorama==0.4.6
comm==0.2.3
debugpy==1.8.19
decorator==5.2.1
executing==2.2.1
ipykernel==7.1.0
ipython==9.9.0
ipython_pygments_lexers==1.1.1
jedi==0.19.2
joblib==1.5.3
jupyter_client==8.8.0
jupyter_core==5.9.1
matplotlib-inline==0.2.1
nest-asyncio==1.6.0
nltk==3.9.2
numpy==2.4.1
packaging==25.0
pandas==2.3.3
parso==0.8.5
platformdirs==4.5.1
prompt_toolkit==3.0.52
psutil==7.2.1
pure_eval==0.2.3
Pygments==2.19.2
python-dateutil==2.9.0.post0
pytz==2025.2
pyzmq==27.1.0
regex==2026.1.15
scikit-learn==1.8.0
scipy==1.17.0
six==1.17.0
stack-data==0.6.3
threadpoolctl==3.6.0
tornado==6.5.4
tqdm==4.67.1
traitlets==5.14.3
tzdata==2025.3
wcwidth==0.2.14
Note: you may need to restart the kernel to use updated packages.


In [5]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\srikr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\srikr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [7]:
from utils.email_read_util import (
    flatten_to_string,
    extract_email_text,
    load
)

from utils.email_read_util import (
    punctuations, stemmer, stopwords  
)
from collections import defaultdict


In [8]:
from utils.blacklist import SPAM_BLACKLIST

In [9]:
df = pd.read_csv("./data/processed_data.csv")
df.head()

,label,subject,email_to,email_from,message
0,1,"Generic Cialis, branded quality@",the00@speedy.uwaterloo.ca,"""Tomas Jacobs"" <RickyAmes@aol.com>",Content-Type: text/html;\nContent-Transfer-Enc...
1,0,Typo in /debian/README,debian-mirrors@lists.debian.org,Yan Morin <yan.morin@savoirfairelinux.com>,"Hi, i've just updated from the gulus and I che..."
2,1,authentic viagra,<the00@plg.uwaterloo.ca>,"""Sheila Crenshaw"" <7stocknews@tractionmarketin...","Content-Type: text/plain;\n\tcharset=""iso-8859..."
3,1,Nice talking with ya,opt4@speedy.uwaterloo.ca,"""Stormy Dempsey"" <vqucsmdfgvsg@ruraltek.com>","Hey Billy, \n\nit was really fun going out the..."
4,1,or trembling; stomach cramps; trouble in sleep...,ktwarwic@speedy.uwaterloo.ca,"""Christi T. Jernigan"" <dcube@totalink.net>",Content-Type: multipart/alternative;\n ...


In [10]:
df.size

377095

In [11]:
df.columns

Index(['label', 'subject', 'email_to', 'email_from', 'message'], dtype='object')

In [12]:
data = df.where((pd.notnull(df)), '')

In [13]:
features = df.drop(columns=['label'])
label = df['label']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(features, label, test_size = 0.3, random_state = 42)

In [35]:
import re
import numpy as np

class NaiveSpamClf(object):
    def __init__(self):
        self.blacklist = None
        self.blacklist_regex = None

    def fit(self, X, y=None):
        """
        X, y kept for API compatibility
        """

        # Assume SPAM_BLACKLIST is already stemmed
        self.blacklist = list(SPAM_BLACKLIST)

        # Compile regex ONCE
        self.blacklist_regex = re.compile(
            r'\b(?:' + '|'.join(map(re.escape, self.blacklist)) + r')\b',
            flags=re.IGNORECASE
        )


    def predict(self, X):
        """
        FAST vectorized prediction
        """

        texts = (
            X['subject'].fillna('').astype(str)
            + " "
            + X['message'].fillna('').astype(str)
        )

        # Vectorized regex match
        is_spam = texts.str.contains(self.blacklist_regex, regex=True)

        return is_spam.astype(int).to_numpy()


In [36]:
nclf = NaiveSpamClf()

In [37]:
nclf.fit(X_train, y_train)

In [38]:
y_pred = nclf.predict(X_test)

In [39]:
from sklearn.metrics import classification_report

print(classification_report(y_pred=y_pred, y_true=y_test))

              precision    recall  f1-score   support

           0       0.40      0.93      0.56      7678
           1       0.88      0.27      0.42     14948

    accuracy                           0.50     22626
   macro avg       0.64      0.60      0.49     22626
weighted avg       0.72      0.50      0.47     22626



In [40]:
X_texts = (
        features['subject'].fillna('') + "\n" + features['message'].fillna('')
)

In [41]:
X_train, X_test, y_train, y_test = train_test_split(X_texts, label)

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

In [44]:
feature_extraction = TfidfVectorizer(min_df = 1, stop_words = 'english', lowercase = True)

X_train = feature_extraction.fit_transform(X_train)
X_test = feature_extraction.transform(X_test)

In [47]:
mb = MultinomialNB()

In [48]:
mb.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [49]:
y_pred = mb.predict(X_test)

In [50]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      6368
           1       0.99      1.00      0.99     12487

    accuracy                           0.99     18855
   macro avg       0.99      0.99      0.99     18855
weighted avg       0.99      0.99      0.99     18855



In [51]:
from sklearn.linear_model import LogisticRegression

In [52]:
lr = LogisticRegression()

In [53]:
lr.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [54]:
y_pred = lr.predict(X_test)

In [55]:
print(classification_report(y_pred=y_pred, y_true=y_test))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99      6368
           1       0.99      1.00      1.00     12487

    accuracy                           0.99     18855
   macro avg       1.00      0.99      0.99     18855
weighted avg       0.99      0.99      0.99     18855

